# Transfer Learning with Pre-trained Embeddings (GloVe)

## What is Transfer Learning in NLP?
Transfer learning involves taking a model (or parts of a model) trained on a large dataset and applying it to a different but related task. 

In the context of processing text with RNNs, the most common form of transfer learning is using **Pre-trained Word Embeddings**.

Instead of learning the meaning of words from scratch (which requires a huge amount of data), we can initialize our `Embedding` layer with vectors learned from massive corpuses (like Wikipedia).

## GloVe (Global Vectors for Word Representation)
GloVe is a popular set of pre-trained word vectors. We will use the 100-dimensional vectors trained on Wikipedia.

In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.random.set_seed(42)

## 1. Downloading GloVe Embeddings
We will download the GloVe dataset. Note: This file is around 800MB.

> **No download needed.** The original lesson fetched the full 822 MB GloVe archive here. This copy ships `glove.6B.100d.imdb10k.txt`: the same 100-dimensional GloVe vectors, kept only for the 9,999 IMDB words this lesson indexes (9,796 of them have a vector). Words outside it stay at zero in the embedding matrix, exactly as they would with the full file.

## 2. Preparing the Data (IMDB)

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200

# Load IMDB data
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
X_train = pad_sequences(X_train, maxlen=MAX_LEN)
X_test = pad_sequences(X_test, maxlen=MAX_LEN)

# Get the word index (word -> integer mapping)
word_index = imdb.get_word_index()

## 3. Creating the Embedding Matrix
We need to create a matrix where row `i` contains the GloVe vector for the word with index `i`.

In [ ]:
embeddings_index = {}
f = open('../../00-data/glove.6B.100d.imdb10k.txt', encoding='utf-8')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

embedding_dim = 100
embedding_matrix = np.zeros((VOCAB_SIZE, embedding_dim))

for word, i in word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            # Words not found in embedding index will be all-zeros.
            embedding_matrix[i] = embedding_vector

## 4. Building the Model with Pre-trained Embeddings
We use the `Embedding` layer but initialize it with our `embedding_matrix` and set `trainable=False` to keep the pre-trained weights frozen.

In [ ]:
model = Sequential()

model.add(Embedding(VOCAB_SIZE, 
                    embedding_dim, 
                    weights=[embedding_matrix], 
                    input_length=MAX_LEN, 
                    trainable=False))

model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

## 5. Fine-Tuning
After training the top layers, we can unfreeze the embedding layer to fine-tune the word vectors for our specific task.

In [ ]:
model.layers[0].trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

history_fine = model.fit(X_train, y_train, epochs=2, batch_size=64, validation_split=0.2)